# Standalone LLM reranker from scratch

This notebook demonstrates the LLM-only pipeline without calling OpenAI and without importing project code. The `mock_llm_score` function acts like a transparent scoring rubric for the kind of JSON output an LLM reranker would produce.

In [1]:
from math import log2
import json

user_profile = {
    "goal": "find a nearby casual place for studying and a light meal",
    "likes": ["coffee", "vietnamese", "quiet", "brunch"],
    "dislikes": ["expensive", "loud", "far away"],
}

candidates = [
    {"id": "b1", "name": "Morning Bean", "text": "quiet coffee shop with bakery snacks", "price": 2, "distance_km": 0.7, "rating": 4.7, "label": 3},
    {"id": "b2", "name": "Pho Corner", "text": "fast vietnamese noodles, casual lunch", "price": 1, "distance_km": 2.2, "rating": 4.5, "label": 3},
    {"id": "b3", "name": "Quiet Study Cafe", "text": "quiet coffee dessert spot with many outlets", "price": 2, "distance_km": 1.1, "rating": 4.1, "label": 2},
    {"id": "b4", "name": "Late Night BBQ", "text": "loud korean bbq for groups", "price": 3, "distance_km": 6.5, "rating": 4.6, "label": 0},
    {"id": "b5", "name": "Campus Banh Mi", "text": "cheap vietnamese sandwich near campus", "price": 1, "distance_km": 0.4, "rating": 4.2, "label": 2},
]


In [2]:
def build_prompt(user_profile, candidates):
    lines = ["Rank these businesses for the user.", f"User goal: {user_profile['goal']}"]
    lines.append("Likes: " + ", ".join(user_profile["likes"]))
    lines.append("Dislikes: " + ", ".join(user_profile["dislikes"]))
    lines.append("Return JSON with business_id, score, and reason.")
    for item in candidates:
        lines.append(f"- {item['id']}: {item['name']}; {item['text']}; price={item['price']}; distance={item['distance_km']}km; rating={item['rating']}")
    return "\n".join(lines)

print(build_prompt(user_profile, candidates))


Rank these businesses for the user.
User goal: find a nearby casual place for studying and a light meal
Likes: coffee, vietnamese, quiet, brunch
Dislikes: expensive, loud, far away
Return JSON with business_id, score, and reason.
- b1: Morning Bean; quiet coffee shop with bakery snacks; price=2; distance=0.7km; rating=4.7
- b2: Pho Corner; fast vietnamese noodles, casual lunch; price=1; distance=2.2km; rating=4.5
- b3: Quiet Study Cafe; quiet coffee dessert spot with many outlets; price=2; distance=1.1km; rating=4.1
- b4: Late Night BBQ; loud korean bbq for groups; price=3; distance=6.5km; rating=4.6
- b5: Campus Banh Mi; cheap vietnamese sandwich near campus; price=1; distance=0.4km; rating=4.2


In [3]:
def mock_llm_score(user_profile, item):
    text = item["text"].lower()
    matched_likes = [word for word in user_profile["likes"] if word in text]
    matched_dislikes = [word for word in user_profile["dislikes"] if word in text]
    score = 0.35 * len(matched_likes)
    score += 0.25 if item["distance_km"] <= 1.5 else 0.05 if item["distance_km"] <= 3.0 else -0.25
    score += 0.15 if item["price"] <= 2 else -0.15
    score += 0.10 * (item["rating"] - 4.0)
    score -= 0.30 * len(matched_dislikes)
    reason = f"matches {matched_likes or ['general quality']}"
    if matched_dislikes:
        reason += f"; penalized for {matched_dislikes}"
    return round(score, 4), reason

def llm_rerank(user_profile, candidates):
    rows = []
    for item in candidates:
        score, reason = mock_llm_score(user_profile, item)
        rows.append({"business_id": item["id"], "name": item["name"], "score": score, "reason": reason, "label": item["label"]})
    return sorted(rows, key=lambda row: row["score"], reverse=True)

ranked = llm_rerank(user_profile, candidates)
print(json.dumps(ranked, indent=2))


[
  {
    "business_id": "b1",
    "name": "Morning Bean",
    "score": 1.17,
    "reason": "matches ['coffee', 'quiet']",
    "label": 3
  },
  {
    "business_id": "b3",
    "name": "Quiet Study Cafe",
    "score": 1.11,
    "reason": "matches ['coffee', 'quiet']",
    "label": 2
  },
  {
    "business_id": "b5",
    "name": "Campus Banh Mi",
    "score": 0.77,
    "reason": "matches ['vietnamese']",
    "label": 2
  },
  {
    "business_id": "b2",
    "name": "Pho Corner",
    "score": 0.6,
    "reason": "matches ['vietnamese']",
    "label": 3
  },
  {
    "business_id": "b4",
    "name": "Late Night BBQ",
    "score": -0.64,
    "reason": "matches ['general quality']; penalized for ['loud']",
    "label": 0
  }
]


In [4]:
def dcg(labels):
    return sum((2 ** label - 1) / log2(rank + 2) for rank, label in enumerate(labels))

def ndcg(rows, k):
    actual = [row["label"] for row in rows[:k]]
    ideal = sorted((row["label"] for row in rows), reverse=True)[:k]
    return dcg(actual) / dcg(ideal) if dcg(ideal) else 0.0

print("ndcg@3", round(ndcg(ranked, 3), 4))
print("ndcg@5", round(ndcg(ranked, 5), 4))


ndcg@3 0.8046
ndcg@5 0.9436
